# Xeno-canto Gathering

Build a reproducible download set using XC query tags.


In [13]:
import json
import sys
import time
from pathlib import Path

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.append(str(repo_root))

from src.config import CONFIG
from src.utils.xeno_canto import get_recordings_data_for_species, download_recordings


In [ ]:
DATA_DIR = Path(CONFIG.paths.data_dir)
RAW_DIR = DATA_DIR / CONFIG.paths.raw_dir
MANIFEST_DIR = DATA_DIR / CONFIG.paths.manifests_dir

for p in [RAW_DIR, MANIFEST_DIR]:
    p.mkdir(parents=True, exist_ok=True)


In [15]:
SPECIES_FILE = Path("../../species_list_small.json")
species_map = json.loads(SPECIES_FILE.read_text(encoding="utf-8"))
species_list = list(species_map.values())
species_list[:5]


[{'common_name': 'European Robin', 'sci_name': 'Erithacus rubecula'},
 {'common_name': 'Eurasian Blackbird', 'sci_name': 'Turdus merula'},
 {'common_name': 'Eurasian Wren', 'sci_name': 'Troglodytes troglodytes'},
 {'common_name': 'Eurasian Blue Tit', 'sci_name': 'Cyanistes caeruleus'},
 {'common_name': 'Great Tit', 'sci_name': 'Parus major'}]

In [16]:
def build_query(sci_name: str) -> str:
    tags = [
        f'sp:"{sci_name}"',
        "grp:birds",
        "area:europe",
        'q:">C"',
        'len:">2"',
        'len:"<60"',
    ]
    return " ".join(tags)

PER_PAGE = 500


## Fetch metadata


In [17]:
MAX_SPECIES = 1

recordings_by_species = {}
species_to_fetch = species_list if MAX_SPECIES is None else species_list[:MAX_SPECIES]

for entry in species_to_fetch:
    sci_name = entry["sci_name"]
    query = build_query(sci_name)
    recs = get_recordings_data_for_species(query, per_page=PER_PAGE)
    recordings_by_species[sci_name] = recs
    print(f"{sci_name}: {len(recs)} recordings")


Erithacus rubecula: 2245 recordings


## Download audio


In [ ]:
MAX_PER_SPECIES = None 

def species_dir_name(sci_name: str) -> str:
    return sci_name.lower().replace(" ", "_")

for sci_name, recs in recordings_by_species.items():
    species_dir = RAW_DIR / species_dir_name(sci_name)
    manifest_csv = MANIFEST_DIR / f"{species_dir.name}.csv"

    entries = write_manifest_only(
        recs,
        manifest_csv_path=manifest_csv,
    )
    print(f"{sci_name}: wrote manifest for {len(entries)} recordings (no downloads)")



Erithacus rubecula: downloaded 2245
